In [64]:
import os
import zipfile
import urllib.request
from google.colab import drive

# 1. Mount Google Drive (ek baar permission deni hogi)
drive.mount('/content/drive')

# 2. Permanent folder path in Google Drive
drive_dir = "/content/drive/MyDrive/CineSearch_Data"
os.makedirs(drive_dir, exist_ok=True)

data_dir = os.path.join(drive_dir, "ml-32m")
zip_path = os.path.join(drive_dir, "ml-32m.zip")
url = "https://files.grouplens.org/datasets/movielens/ml-32m.zip"

# Download & extract ONLY if not already saved in Google Drive
if not os.path.exists(os.path.join(data_dir, "movies.csv")):
    print("Downloading MovieLens 32M dataset to Google Drive (one-time setup)...")
    urllib.request.urlretrieve(url, zip_path)

    print("Extracting dataset in Google Drive...")
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(drive_dir)
    print("Saved permanently to Google Drive!")
else:
    print("Dataset already exists in Google Drive! Skipping download.")

print("Files available:", os.listdir(data_dir))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset already exists in Google Drive! Skipping download.
Files available: ['tags.csv', 'links.csv', 'README.txt', 'checksums.txt', 'ratings.csv', 'movies.csv']


In [65]:
### Dataset Preparation — Loading Movie Catalog Metadata
import pandas as pd
import os

movies_path = "/content/drive/MyDrive/CineSearch_Data/ml-32m/movies.csv"

movies = pd.read_csv(movies_path)

print("Total movies:", len(movies))
print("\nFirst 10 movies:")
display(movies.head(10))

print("\nColumn names:")
print(movies.columns.tolist())

Total movies: 87585

First 10 movies:


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy
5,6,Heat (1995),Action|Crime|Thriller
6,7,Sabrina (1995),Comedy|Romance
7,8,Tom and Huck (1995),Adventure|Children
8,9,Sudden Death (1995),Action
9,10,GoldenEye (1995),Action|Adventure|Thriller



Column names:
['movieId', 'title', 'genres']


In [66]:
### Database-Derived Ranking Features & Average-Rating Scoring
import pandas as pd

ratings_path = "/content/drive/MyDrive/CineSearch_Data/ml-32m/ratings.csv"

rating_parts = []

for chunk in pd.read_csv(ratings_path, chunksize=500000):
    summary = chunk.groupby("movieId").agg(
        rating_sum=("rating", "sum"),
        rating_count=("rating", "count")
    )
    rating_parts.append(summary)

all_ratings = pd.concat(rating_parts)

movie_stats = all_ratings.groupby("movieId").agg(
    rating_sum=("rating_sum", "sum"),
    rating_count=("rating_count", "sum")
).reset_index()

movie_stats["average_rating"] = (
    movie_stats["rating_sum"] / movie_stats["rating_count"]
)

movies = movies.merge(movie_stats, on="movieId", how="left")

movies["rating_count"] = movies["rating_count"].fillna(0)
movies["average_rating"] = movies["average_rating"].fillna(0)

print("Rating analysis completed!")
print("Movies with ratings:", (movies["rating_count"] > 0).sum())

display(
    movies[["title", "genres", "average_rating", "rating_count"]]
    .sort_values("rating_count", ascending=False)
    .head(10)
)

Rating analysis completed!
Movies with ratings: 84432


,title,genres,average_rating,rating_count
314,"Shawshank Redemption, The (1994)",Crime|Drama,4.404614,102929.0
351,Forrest Gump (1994),Comedy|Drama|Romance|War,4.052744,100296.0
292,Pulp Fiction (1994),Comedy|Crime|Drama|Thriller,4.196969,98409.0
2480,"Matrix, The (1999)",Action|Sci-Fi|Thriller,4.156437,93808.0
585,"Silence of the Lambs, The (1991)",Crime|Horror|Thriller,4.148367,90330.0
257,Star Wars: Episode IV - A New Hope (1977),Action|Adventure|Sci-Fi,4.099824,85010.0
2867,Fight Club (1999),Action|Crime|Drama|Thriller,4.228780,77332.0
475,Jurassic Park (1993),Action|Adventure|Sci-Fi|Thriller,3.698623,75233.0
522,Schindler's List (1993),Drama|War,4.236990,73849.0
4888,"Lord of the Rings: The Fellowship of the Ring,...",Adventure|Fantasy,4.092134,73122.0


In [67]:
### Popularity Scoring, Feature Normalization & Composite Ranking
import numpy as np

# Normalize popularity using log to reduce the effect of extremely popular movies
movies["popularity_score"] = np.log1p(movies["rating_count"])

max_popularity = movies["popularity_score"].max()

if max_popularity > 0:
    movies["popularity_score"] = (
        movies["popularity_score"] / max_popularity
    )

# Normalize average rating to a score between 0 and 1
movies["rating_score"] = movies["average_rating"] / 5.0

# Composite ranking: combine popularity and average rating
movies["composite_score"] = (
    0.5 * movies["popularity_score"]
    + 0.5 * movies["rating_score"]
)

# Rank movies from highest to lowest composite score
movies["rank"] = (
    movies["composite_score"]
    .rank(method="min", ascending=False)
    .astype(int)
)

top_movies = movies.sort_values(
    "composite_score", ascending=False
).head(10)

print("Top 10 Movies by Composite Ranking")
display(
    top_movies[
        ["rank", "title", "genres", "average_rating",
         "rating_count", "popularity_score", "composite_score"]
    ]
)

Top 10 Movies by Composite Ranking


,rank,title,genres,average_rating,rating_count,popularity_score,composite_score
314,1,"Shawshank Redemption, The (1994)",Crime|Drama,4.404614,102929.0,1.000000,0.940461
292,2,Pulp Fiction (1994),Comedy|Crime|Drama|Thriller,4.196969,98409.0,0.996109,0.917751
840,3,"Godfather, The (1972)",Crime|Drama,4.317030,66440.0,0.962074,0.912740
2480,4,"Matrix, The (1999)",Action|Sci-Fi|Thriller,4.156437,93808.0,0.991961,0.911624
2867,5,Fight Club (1999),Action|Crime|Drama|Thriller,4.228780,77332.0,0.975227,0.910491
522,6,Schindler's List (1993),Drama|War,4.236990,73849.0,0.971234,0.909316
585,7,"Silence of the Lambs, The (1991)",Crime|Horror|Thriller,4.148367,90330.0,0.988687,0.909180
49,8,"Usual Suspects, The (1995)",Crime|Mystery|Thriller,4.265070,67750.0,0.963766,0.908390
351,9,Forrest Gump (1994),Comedy|Drama|Romance|War,4.052744,100296.0,0.997755,0.904152
257,10,Star Wars: Episode IV - A New Hope (1977),Action|Adventure|Sci-Fi,4.099824,85010.0,0.983428,0.901696


In [68]:
### TF-IDF Ranking & Field-Weighted Ranking
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Prepare movie title and genre text
titles = movies["title"].fillna("").astype(str)
genres = movies["genres"].fillna("").astype(str)

# TF-IDF for movie titles
title_vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2)
)
title_matrix = title_vectorizer.fit_transform(titles)

# TF-IDF for movie genres
genre_vectorizer = TfidfVectorizer(
    token_pattern=r"(?u)\b\w+\b"
)
genre_matrix = genre_vectorizer.fit_transform(genres)

def search_movies(query, top_n=10):
    query = query.strip()

    if not query:
        print("Please enter a search query.")
        return

    title_query = title_vectorizer.transform([query])
    genre_query = genre_vectorizer.transform([query])

    title_scores = cosine_similarity(
        title_query, title_matrix
    ).flatten()

    genre_scores = cosine_similarity(
        genre_query, genre_matrix
    ).flatten()

    # Field-weighted relevance score
    search_scores = (
        0.7 * title_scores + 0.3 * genre_scores
    )

    results = movies.copy()
    results["search_score"] = search_scores

    results = results[results["search_score"] > 0]
    results = results.sort_values(
        "search_score", ascending=False
    ).head(top_n)

    print("Search results for:", query)

    if results.empty:
        print("No matching movies found.")
    else:
        display(
            results[
                ["title", "genres", "average_rating",
                 "rating_count", "search_score"]
            ]
        )

print("TF-IDF Search Ranking is ready!")
search_movies("Batman", 10)

TF-IDF Search Ranking is ready!
Search results for: Batman


,title,genres,average_rating,rating_count,search_score
46768,Batman & Bill (2017),Documentary,3.814815,27.0,0.397751
80120,The Batman (2022),Action|Crime|Drama,3.672605,3340.0,0.394296
584,Batman (1989),Action|Crime|Thriller,3.409920,49789.0,0.385933
8616,Batman (1966),Action|Adventure|Comedy,3.214971,1042.0,0.381484
17362,Batman (1943),Action|Adventure|Crime|Sci-Fi|Thriller,2.983333,30.0,0.375317
48161,Batman Beyond: The Movie (1999),Action|Animation|Children,3.500000,59.0,0.311715
72228,Heart of Batman (2018),Documentary,3.833333,3.0,0.296398
53894,Batman Ninja (2018),Action|Animation,2.937500,192.0,0.294648
151,Batman Forever (1995),Action|Adventure|Comedy|Crime,2.862687,35288.0,0.287450
1506,Batman & Robin (1997),Action|Adventure|Fantasy|Thriller,2.161005,11686.0,0.286160


In [69]:
### Interactive Search Interface — TF-IDF & Field-Weighted Ranking Demo
import ipywidgets as widgets
from IPython.display import display, clear_output

movie_input = widgets.Text(
    placeholder="e.g. Batman, Avatar, Comedy",
    description="Search:",
    layout=widgets.Layout(width="70%")
)

search_button = widgets.Button(
    description="Search",
    button_style="primary"
)

results_output = widgets.Output()

def run_movie_search(button=None):
    with results_output:
        clear_output(wait=True)
        query = movie_input.value.strip()

        if query:
            search_movies(query, 10)
        else:
            print("Please enter a movie name or genre.")

search_button.on_click(run_movie_search)
movie_input.on_submit(run_movie_search)

display(widgets.VBox([
    widgets.HTML("<h2>CineSearch</h2><p>Search movies by title or genre</p>"),
    widgets.HBox([movie_input, search_button]),
    results_output
]))

In [70]:
### Popularity-Based Recommendation
# 1. Recommend popular movies
def recommend_popular(top_n=10):
    results = movies.sort_values(
        "composite_score", ascending=False
    ).head(top_n)

    print("Popular Movie Recommendations")
    display(results[
        ["title", "genres", "average_rating",
         "rating_count", "composite_score"]
    ])

# 2. Recommend movies by genre
def recommend_by_genre(genre, top_n=10):
    genre = genre.strip().lower()

    matching = movies[
        movies["genres"].str.lower().str.split("|").apply(
            lambda genres: genre in genres
        )
    ]

    results = matching.sort_values(
        "composite_score", ascending=False
    ).head(top_n)

    print("Recommendations for genre:", genre.title())

    if results.empty:
        print("No movies found for this genre.")
    else:
        display(results[
            ["title", "genres", "average_rating",
             "rating_count", "composite_score"]
        ])

print("Recommendation system is ready!")

recommend_popular(10)
recommend_by_genre("Comedy", 10)

Recommendation system is ready!
Popular Movie Recommendations


,title,genres,average_rating,rating_count,composite_score
314,"Shawshank Redemption, The (1994)",Crime|Drama,4.404614,102929.0,0.940461
292,Pulp Fiction (1994),Comedy|Crime|Drama|Thriller,4.196969,98409.0,0.917751
840,"Godfather, The (1972)",Crime|Drama,4.317030,66440.0,0.912740
2480,"Matrix, The (1999)",Action|Sci-Fi|Thriller,4.156437,93808.0,0.911624
2867,Fight Club (1999),Action|Crime|Drama|Thriller,4.228780,77332.0,0.910491
522,Schindler's List (1993),Drama|War,4.236990,73849.0,0.909316
585,"Silence of the Lambs, The (1991)",Crime|Horror|Thriller,4.148367,90330.0,0.909180
49,"Usual Suspects, The (1995)",Crime|Mystery|Thriller,4.265070,67750.0,0.908390
351,Forrest Gump (1994),Comedy|Drama|Romance|War,4.052744,100296.0,0.904152
257,Star Wars: Episode IV - A New Hope (1977),Action|Adventure|Sci-Fi,4.099824,85010.0,0.901696


Recommendations for genre: Comedy


,title,genres,average_rating,rating_count,composite_score
292,Pulp Fiction (1994),Comedy|Crime|Drama|Thriller,4.196969,98409.0,0.917751
351,Forrest Gump (1994),Comedy|Drama|Romance|War,4.052744,100296.0,0.904152
600,Fargo (1996),Comedy|Crime|Drama|Thriller,4.116533,58031.0,0.886828
1108,Monty Python and the Holy Grail (1975),Adventure|Comedy|Fantasy,4.135633,46508.0,0.879149
1167,"Princess Bride, The (1987)",Action|Adventure|Comedy|Fantasy|Romance,4.119109,46957.0,0.877913
1237,Back to the Future (1985),Adventure|Comedy|Sci-Fi,3.969149,61879.0,0.874871
0,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,3.897438,68997.0,0.872417
4868,"Amelie (Fabuleux destin d'Amélie Poulain, Le) ...",Comedy|Romance,4.082169,43459.0,0.870865
734,Dr. Strangelove or: How I Learned to Stop Worr...,Comedy|War,4.202680,32830.0,0.870766
2233,Life Is Beautiful (La Vita è bella) (1997),Comedy|Drama|Romance|War,4.150374,29819.0,0.861369


In [71]:
recommend_by_genre("Action", 10)
recommend_by_genre("Comedy", 10)
recommend_by_genre("Horror", 10)
recommend_by_genre("Sci-Fi", 10)

Recommendations for genre: Action


,title,genres,average_rating,rating_count,composite_score
2480,"Matrix, The (1999)",Action|Sci-Fi|Thriller,4.156437,93808.0,0.911624
2867,Fight Club (1999),Action|Crime|Drama|Thriller,4.228780,77332.0,0.910491
257,Star Wars: Episode IV - A New Hope (1977),Action|Adventure|Sci-Fi,4.099824,85010.0,0.901696
1166,Star Wars: Episode V - The Empire Strikes Back...,Action|Adventure|Sci-Fi,4.130352,72151.0,0.897644
12223,"Dark Knight, The (2008)",Action|Crime|Drama|IMAX,4.173349,59334.0,0.893472
1168,Raiders of the Lost Ark (Indiana Jones and the...,Action|Adventure,4.111893,67408.0,0.892853
7029,"Lord of the Rings: The Return of the King, The...",Action|Adventure|Drama|Fantasy,4.094360,67449.0,0.891126
14939,Inception (2010),Action|Crime|Drama|Mystery|Sci-Fi|Thriller|IMAX,4.157170,57931.0,0.890817
108,Braveheart (1995),Action|Drama|War,3.988630,69482.0,0.881839
1179,Star Wars: Episode VI - Return of the Jedi (1983),Action|Adventure|Sci-Fi,3.991555,67496.0,0.880876


Recommendations for genre: Comedy


,title,genres,average_rating,rating_count,composite_score
292,Pulp Fiction (1994),Comedy|Crime|Drama|Thriller,4.196969,98409.0,0.917751
351,Forrest Gump (1994),Comedy|Drama|Romance|War,4.052744,100296.0,0.904152
600,Fargo (1996),Comedy|Crime|Drama|Thriller,4.116533,58031.0,0.886828
1108,Monty Python and the Holy Grail (1975),Adventure|Comedy|Fantasy,4.135633,46508.0,0.879149
1167,"Princess Bride, The (1987)",Action|Adventure|Comedy|Fantasy|Romance,4.119109,46957.0,0.877913
1237,Back to the Future (1985),Adventure|Comedy|Sci-Fi,3.969149,61879.0,0.874871
0,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,3.897438,68997.0,0.872417
4868,"Amelie (Fabuleux destin d'Amélie Poulain, Le) ...",Comedy|Romance,4.082169,43459.0,0.870865
734,Dr. Strangelove or: How I Learned to Stop Worr...,Comedy|War,4.202680,32830.0,0.870766
2233,Life Is Beautiful (La Vita è bella) (1997),Comedy|Drama|Romance|War,4.150374,29819.0,0.861369


Recommendations for genre: Horror


,title,genres,average_rating,rating_count,composite_score
585,"Silence of the Lambs, The (1991)",Crime|Horror|Thriller,4.148367,90330.0,0.909180
2670,"Sixth Sense, The (1999)",Drama|Horror|Mystery,4.003210,57010.0,0.874727
1183,Alien (1979),Horror|Sci-Fi,4.068401,45211.0,0.871201
1225,"Shining, The (1980)",Horror,4.036439,38640.0,0.861201
1170,Aliens (1986),Action|Adventure|Horror|Sci-Fi,4.007259,38846.0,0.858513
1188,Psycho (1960),Crime|Horror,4.062512,27027.0,0.848324
8161,Shaun of the Dead (2004),Comedy|Horror,3.878019,26914.0,0.829693
1351,Jaws (1975),Action|Horror,3.785016,29265.0,0.824020
2197,"Thing, The (1982)",Action|Horror|Sci-Fi|Thriller,3.935967,14719.0,0.809345
6380,28 Days Later (2002),Action|Horror|Sci-Fi,3.772137,21390.0,0.809153


Recommendations for genre: Sci-Fi


,title,genres,average_rating,rating_count,composite_score
2480,"Matrix, The (1999)",Action|Sci-Fi|Thriller,4.156437,93808.0,0.911624
257,Star Wars: Episode IV - A New Hope (1977),Action|Adventure|Sci-Fi,4.099824,85010.0,0.901696
1166,Star Wars: Episode V - The Empire Strikes Back...,Action|Adventure|Sci-Fi,4.130352,72151.0,0.897644
14939,Inception (2010),Action|Crime|Drama|Mystery|Sci-Fi|Thriller|IMAX,4.157170,57931.0,0.890817
1179,Star Wars: Episode VI - Return of the Jedi (1983),Action|Adventure|Sci-Fi,3.991555,67496.0,0.880876
581,Terminator 2: Judgment Day (1991),Action|Sci-Fi,3.962045,68383.0,0.878490
536,Blade Runner (1982),Action|Sci-Fi|Thriller,4.110005,46289.0,0.876382
1237,Back to the Future (1985),Adventure|Comedy|Sci-Fi,3.969149,61879.0,0.874871
1183,Alien (1979),Horror|Sci-Fi,4.068401,45211.0,0.871201
7237,Eternal Sunshine of the Spotless Mind (2004),Drama|Romance|Sci-Fi,4.065294,44430.0,0.870135


In [72]:
### Interactive Recommendation Interface — Popularity & Genre-Based Demo
import ipywidgets as widgets
from IPython.display import display, clear_output

# Check whether the movie dataset is loaded
if "movies" not in globals():
    raise NameError(
        "movies dataset is not loaded. "
        "Run the previous cell that creates the movies DataFrame first."
    )

# Check required columns
required_columns = [
    "title", "genres", "average_rating",
    "rating_count", "composite_score"
]

missing_columns = [
    col for col in required_columns
    if col not in movies.columns
]

if missing_columns:
    raise ValueError(
        f"Missing columns in movies: {missing_columns}. "
        "Run the earlier data preparation cell first."
    )

# Get available genres from the dataset
all_genres = sorted({
    genre
    for genre_list in movies["genres"].dropna()
    for genre in genre_list.split("|")
    if genre and genre != "(no genres listed)"
})

if not all_genres:
    raise ValueError(
        "No genres found. Check the movies dataset."
    )

genre_dropdown = widgets.Dropdown(
    options=all_genres,
    description="Choose genre:",
    layout=widgets.Layout(width="350px")
)

recommend_button = widgets.Button(
    description="Recommend Movies",
    button_style="primary"
)

recommend_output = widgets.Output()

def show_recommendations(button):
    with recommend_output:
        clear_output(wait=True)

        selected_genre = genre_dropdown.value

        matching_movies = movies[
            movies["genres"].fillna("").apply(
                lambda x: selected_genre in x.split("|")
            )
        ]

        results = matching_movies.sort_values(
            "composite_score",
            ascending=False
        ).head(10)

        print("Recommended movies:", selected_genre)

        if results.empty:
            print("No movies found for this genre.")
        else:
            display(results[required_columns])

recommend_button.on_click(show_recommendations)

display(widgets.VBox([
    widgets.HTML("<h3>🎬 CineSearch Recommendations</h3>"),
    genre_dropdown,
    recommend_button,
    recommend_output
]))

In [73]:
###  Term Frequency (TF) Calculation
import re
from collections import Counter
import pandas as pd

def tokenize(text):
    # Clean text and extract lowercase alphanumeric tokens
    return re.findall(r"\b\w+\b", str(text).lower())

def calculate_tf(document):
    tokens = tokenize(document)
    total_terms = len(tokens)
    if total_terms == 0:
        return {}
    term_counts = Counter(tokens)
    # TF(t, d) = (Number of times term t appears in document d) / (Total terms in document d)
    tf_scores = {term: round(count / total_terms, 4) for term, count in term_counts.items()}
    return tf_scores

# Demonstrate TF on sample movie titles + genres combined
sample_docs = (movies["title"] + " " + movies["genres"].str.replace("|", " ", regex=False)).head(5)

tf_records = []
for idx, doc in enumerate(sample_docs):
    tf_dict = calculate_tf(doc)
    tf_records.append({
        "movieId": movies.loc[idx, "movieId"],
        "Document (Title + Genres)": doc,
        "Total Tokens": len(tokenize(doc)),
        "Term Frequencies (TF)": tf_dict
    })

print("--- Explicit Term Frequency (TF) Calculation ---")
display(pd.DataFrame(tf_records))

--- Explicit Term Frequency (TF) Calculation ---


,movieId,Document (Title + Genres),Total Tokens,Term Frequencies (TF)
0,1,Toy Story (1995) Adventure Animation Children ...,8,"{'toy': 0.125, 'story': 0.125, '1995': 0.125, ..."
1,2,Jumanji (1995) Adventure Children Fantasy,5,"{'jumanji': 0.2, '1995': 0.2, 'adventure': 0.2..."
2,3,Grumpier Old Men (1995) Comedy Romance,6,"{'grumpier': 0.1667, 'old': 0.1667, 'men': 0.1..."
3,4,Waiting to Exhale (1995) Comedy Drama Romance,7,"{'waiting': 0.1429, 'to': 0.1429, 'exhale': 0...."
4,5,Father of the Bride Part II (1995) Comedy,8,"{'father': 0.125, 'of': 0.125, 'the': 0.125, '..."


In [74]:
###  Inverse Document Frequency (IDF) Calculation
import math

def calculate_idf(corpus):
    N = len(corpus)
    doc_freq = Counter()

    for doc in corpus:
        unique_terms = set(tokenize(doc))
        for term in unique_terms:
            doc_freq[term] += 1

    # Smooth IDF formula: log((N + 1) / (df(t) + 1)) + 1
    idf_scores = {
        term: round(math.log((N + 1) / (df + 1)) + 1, 4)
        for term, df in doc_freq.items()
    }
    return idf_scores, doc_freq

# Compute IDF across the entire movie catalog (Titles + Genres)
full_corpus = (movies["title"].fillna("") + " " + movies["genres"].fillna("").str.replace("|", " ", regex=False)).tolist()
idf_map, df_map = calculate_idf(full_corpus)

# Show IDF for high-frequency vs rare terms
sample_terms = ["the", "comedy", "drama", "action", "batman", "matrix", "inception", "thriller"]
idf_table = []
for term in sample_terms:
    if term in idf_map:
        idf_table.append({
            "Term": term,
            "Document Frequency (df)": df_map[term],
            "Total Documents (N)": len(full_corpus),
            "IDF Score": idf_map[term]
        })

print("--- Explicit Inverse Document Frequency (IDF) across Full Catalog ---")
display(pd.DataFrame(idf_table).sort_values("IDF Score", ascending=False))

--- Explicit Inverse Document Frequency (IDF) across Full Catalog ---


,Term,Document Frequency (df),Total Documents (N),IDF Score
6,inception,1,87585,11.6872
5,matrix,14,87585,9.6723
4,batman,43,87585,8.5962
3,action,9692,87585,3.2012
7,thriller,11827,87585,3.0022
1,comedy,23139,87585,2.3311
0,the,24907,87585,2.2574
2,drama,34183,87585,1.9409


In [75]:
### Structured Ranking Experiments
# Compare how different feature weights affect top-ranked movies
experiments = [
    {"Experiment": "Exp 1: Pure Popularity",      "w_pop": 1.0, "w_rat": 0.0},
    {"Experiment": "Exp 2: Popularity Dominant",  "w_pop": 0.8, "w_rat": 0.2},
    {"Experiment": "Exp 3: Balanced Composite",   "w_pop": 0.5, "w_rat": 0.5},
    {"Experiment": "Exp 4: Rating Dominant",      "w_pop": 0.2, "w_rat": 0.8},
    {"Experiment": "Exp 5: Pure Average Rating",  "w_pop": 0.0, "w_rat": 1.0},
]

summary_comparison = []

for exp in experiments:
    name = exp["Experiment"]
    w_p, w_r = exp["w_pop"], exp["w_rat"]

    score_col = w_p * movies["popularity_score"] + w_r * movies["rating_score"]
    ranked_df = movies.assign(exp_score=score_col).sort_values("exp_score", ascending=False).head(5)

    print(f"\n=== {name} (Pop Weight: {w_p}, Rating Weight: {w_r}) ===")
    display(ranked_df[["title", "genres", "average_rating", "rating_count", "exp_score"]])

    summary_comparison.append({
        "Experiment": name,
        "Top #1 Movie": ranked_df.iloc[0]["title"],
        "Avg Rating of Top 5": round(ranked_df["average_rating"].mean(), 3),
        "Avg Rating Count of Top 5": int(ranked_df["rating_count"].mean())
    })

print("\n--- Structured Ranking Experiments Summary ---")
display(pd.DataFrame(summary_comparison))


=== Exp 1: Pure Popularity (Pop Weight: 1.0, Rating Weight: 0.0) ===


,title,genres,average_rating,rating_count,exp_score
314,"Shawshank Redemption, The (1994)",Crime|Drama,4.404614,102929.0,1.000000
351,Forrest Gump (1994),Comedy|Drama|Romance|War,4.052744,100296.0,0.997755
292,Pulp Fiction (1994),Comedy|Crime|Drama|Thriller,4.196969,98409.0,0.996109
2480,"Matrix, The (1999)",Action|Sci-Fi|Thriller,4.156437,93808.0,0.991961
585,"Silence of the Lambs, The (1991)",Crime|Horror|Thriller,4.148367,90330.0,0.988687



=== Exp 2: Popularity Dominant (Pop Weight: 0.8, Rating Weight: 0.2) ===


,title,genres,average_rating,rating_count,exp_score
314,"Shawshank Redemption, The (1994)",Crime|Drama,4.404614,102929.0,0.976185
292,Pulp Fiction (1994),Comedy|Crime|Drama|Thriller,4.196969,98409.0,0.964766
351,Forrest Gump (1994),Comedy|Drama|Romance|War,4.052744,100296.0,0.960314
2480,"Matrix, The (1999)",Action|Sci-Fi|Thriller,4.156437,93808.0,0.959826
585,"Silence of the Lambs, The (1991)",Crime|Horror|Thriller,4.148367,90330.0,0.956885



=== Exp 3: Balanced Composite (Pop Weight: 0.5, Rating Weight: 0.5) ===


,title,genres,average_rating,rating_count,exp_score
314,"Shawshank Redemption, The (1994)",Crime|Drama,4.404614,102929.0,0.940461
292,Pulp Fiction (1994),Comedy|Crime|Drama|Thriller,4.196969,98409.0,0.917751
840,"Godfather, The (1972)",Crime|Drama,4.317030,66440.0,0.912740
2480,"Matrix, The (1999)",Action|Sci-Fi|Thriller,4.156437,93808.0,0.911624
2867,Fight Club (1999),Action|Crime|Drama|Thriller,4.228780,77332.0,0.910491


... (2 outputs left out)


=== Exp 5: Pure Average Rating (Pop Weight: 0.0, Rating Weight: 1.0) ===


,title,genres,average_rating,rating_count,exp_score
75188,The Science of Sleep (2016),Documentary,5.0,1.0,1.0
75001,Once Upon a Main Street (2020),Drama,5.0,1.0,1.0
75176,Do You Know the Muffin Man? (1989),Drama,5.0,1.0,1.0
29151,Temple of the White Elephant (1964),Action|Adventure|Drama,5.0,1.0,1.0
75175,One Summer Night (2016),Drama,5.0,1.0,1.0



--- Structured Ranking Experiments Summary ---


,Experiment,Top #1 Movie,Avg Rating of Top 5,Avg Rating Count of Top 5
0,Exp 1: Pure Popularity,"Shawshank Redemption, The (1994)",4.192,97154
1,Exp 2: Popularity Dominant,"Shawshank Redemption, The (1994)",4.192,97154
2,Exp 3: Balanced Composite,"Shawshank Redemption, The (1994)",4.261,87783
3,Exp 4: Rating Dominant,"Shawshank Redemption, The (1994)",4.290,77660
4,Exp 5: Pure Average Rating,The Science of Sleep (2016),5.000,1


In [76]:
#  Train/Test ProtocolRecommendation
import numpy as np

# Stream through the ENTIRE 32M ratings file in chunks to extract a clean, active user cohort
# (Users with 30 to 500 ratings so they have enough history for train/test without memory overflow)
print("Scanning full 32M ratings dataset for active user cohort...")

user_rating_counts = Counter()
for chunk in pd.read_csv(ratings_path, usecols=["userId"], chunksize=1_000_000):
    user_rating_counts.update(chunk["userId"].value_counts().to_dict())

# Select 3,000 active users who have between 30 and 300 ratings
eligible_users = [
    uid for uid, cnt in user_rating_counts.items()
    if 30 <= cnt <= 300
][:3000]
eligible_user_set = set(eligible_users)

# Collect all interactions for these 3,000 users across the full 32M file
filtered_chunks = []
for chunk in pd.read_csv(ratings_path, chunksize=1_000_000):
    sub = chunk[chunk["userId"].isin(eligible_user_set)]
    if not sub.empty:
        filtered_chunks.append(sub)

interactions_df = pd.concat(filtered_chunks, ignore_index=True)

# Temporal / Leave-Last-20%-Out Train/Test Split per user
interactions_df = interactions_df.sort_values(["userId", "timestamp"]).reset_index(drop=True)
interactions_df["user_seq"] = interactions_df.groupby("userId").cumcount()
interactions_df["user_total"] = interactions_df.groupby("userId")["movieId"].transform("count")
interactions_df["split_ratio"] = interactions_df["user_seq"] / interactions_df["user_total"]

train_df = interactions_df[interactions_df["split_ratio"] < 0.8].copy()
test_df = interactions_df[interactions_df["split_ratio"] >= 0.8].copy()

print("Train/Test Protocol Ready!")
print(f"Total Selected Users : {len(eligible_user_set)}")
print(f"Total Interactions   : {len(interactions_df)}")
print(f"Train Set (80%)      : {len(train_df)} interactions")
print(f"Test Set (20%)       : {len(test_df)} interactions")

Scanning full 32M ratings dataset for active user cohort...
Train/Test Protocol Ready!
Total Selected Users : 3000
Total Interactions   : 388313
Train Set (80%)      : 311853 interactions
Test Set (20%)       : 76460 interactions


In [77]:
###  User Genre-Affinity Recommendation
# Consider ratings >= 3.5 as positive user preferences in the training set
positive_train = train_df[train_df["rating"] >= 3.5].merge(
    movies[["movieId", "title", "genres", "composite_score"]], on="movieId"
)

def build_user_genre_profile(user_id):
    user_likes = positive_train[positive_train["userId"] == user_id]
    genre_scores = Counter()

    for _, row in user_likes.iterrows():
        for g in str(row["genres"]).split("|"):
            if g and g != "(no genres listed)":
                # Weight genre preference by the rating given by the user
                genre_scores[g] += row["rating"]

    total_score = sum(genre_scores.values())
    if total_score == 0:
        return {}
    return {g: round(val / total_score, 4) for g, val in genre_scores.items()}

def recommend_genre_affinity(user_id, top_n=10):
    profile = build_user_genre_profile(user_id)
    seen_movies = set(train_df[train_df["userId"] == user_id]["movieId"])

    candidates = movies[~movies["movieId"].isin(seen_movies)].copy()
    if not profile:
        return candidates.sort_values("composite_score", ascending=False).head(top_n)

    def calc_affinity(genres_str):
        g_list = str(genres_str).split("|")
        return sum(profile.get(g, 0.0) for g in g_list)

    candidates["genre_affinity"] = candidates["genres"].apply(calc_affinity)
    # Combine personalized genre affinity (70%) with overall movie quality (30%)
    candidates["personalized_score"] = 0.7 * candidates["genre_affinity"] + 0.3 * candidates["composite_score"]

    return candidates.sort_values("personalized_score", ascending=False).head(top_n)

# Demonstrate on a specific user
demo_user = eligible_users[0]
print(f"User {demo_user} Learned Genre Profile:")
print(build_user_genre_profile(demo_user))

print(f"\nTop 10 Genre-Affinity Recommendations for User {demo_user}:")
display(recommend_genre_affinity(demo_user, 10)[["movieId", "title", "genres", "genre_affinity", "personalized_score"]])

User 1016 Learned Genre Profile:
{'Comedy': 0.2264, 'Drama': 0.1757, 'Romance': 0.1155, 'Adventure': 0.084, 'Animation': 0.0205, 'Children': 0.0256, 'Fantasy': 0.0493, 'Action': 0.0707, 'Sci-Fi': 0.0406, 'Thriller': 0.0493, 'Crime': 0.0644, 'Mystery': 0.0146, 'Musical': 0.0383, 'IMAX': 0.01, 'War': 0.0078, 'Horror': 0.0037, 'Documentary': 0.0037}

Top 10 Genre-Affinity Recommendations for User 1016:


,movieId,title,genres,genre_affinity,personalized_score
1818,1907,Mulan (1998),Adventure|Animation|Children|Comedy|Drama|Musi...,0.6860,0.717964
4851,4956,"Stunt Man, The (1980)",Action|Adventure|Comedy|Drama|Romance|Thriller,0.7216,0.702119
9179,27344,Revolutionary Girl Utena: Adolescence of Utena...,Action|Adventure|Animation|Comedy|Drama|Fantas...,0.7421,0.692173
4615,4719,Osmosis Jones (2001),Action|Animation|Comedy|Crime|Drama|Romance|Th...,0.7225,0.688957
34202,144324,Once Upon a Time (2008),Action|Adventure|Comedy|Crime|Drama|Romance|Th...,0.7860,0.683404
579,587,Ghost (1990),Comedy|Drama|Fantasy|Romance|Thriller,0.6162,0.669571
1823,1912,Out of Sight (1998),Comedy|Crime|Drama|Romance|Thriller,0.6313,0.669534
10832,45672,Click (2006),Adventure|Comedy|Drama|Fantasy|Romance,0.6509,0.658294
6780,6902,Interstate 60 (2002),Adventure|Comedy|Drama|Fantasy|Mystery|Sci-Fi|...,0.6399,0.658286
8679,26236,"White Sun of the Desert, The (Beloe solntse pu...",Action|Adventure|Comedy|Drama|Romance|War,0.6801,0.656544


In [78]:
###  Item Co-occurrence Recommendation
from collections import defaultdict

# Map each movie to the set of users who liked it in the training set
movie_to_users = positive_train.groupby("movieId")["userId"].apply(set).to_dict()
user_to_movies = positive_train.groupby("userId")["movieId"].apply(list).to_dict()

def get_cooccurrence_recommendations(seed_movie_id, top_n=10):
    if seed_movie_id not in movie_to_users:
        return pd.DataFrame()

    seed_users = movie_to_users[seed_movie_id]
    co_counts = Counter()

    for uid in seed_users:
        for other_mid in user_to_movies.get(uid, []):
            if other_mid != seed_movie_id:
                co_counts[other_mid] += 1

    top_co = co_counts.most_common(top_n)
    co_df = pd.DataFrame(top_co, columns=["movieId", "co_occurrence_count"])
    return co_df.merge(movies[["movieId", "title", "genres", "average_rating"]], on="movieId")

# Pick a popular movie from train set as seed
seed_mid = positive_train["movieId"].value_counts().index[0]
seed_title = movies.loc[movies["movieId"] == seed_mid, "title"].values[0]

print(f"Item Co-occurrence Recommendations for '{seed_title}' (Users who liked this also liked):")
display(get_cooccurrence_recommendations(seed_mid, 10))

Item Co-occurrence Recommendations for 'Shawshank Redemption, The (1994)' (Users who liked this also liked):


,movieId,co_occurrence_count,title,genres,average_rating
0,296,870,Pulp Fiction (1994),Comedy|Crime|Drama|Thriller,4.196969
1,356,851,Forrest Gump (1994),Comedy|Drama|Romance|War,4.052744
2,593,805,"Silence of the Lambs, The (1991)",Crime|Horror|Thriller,4.148367
3,2571,759,"Matrix, The (1999)",Action|Sci-Fi|Thriller,4.156437
4,527,689,Schindler's List (1993),Drama|War,4.236990
5,50,679,"Usual Suspects, The (1995)",Crime|Mystery|Thriller,4.265070
6,2959,668,Fight Club (1999),Action|Crime|Drama|Thriller,4.228780
7,260,584,Star Wars: Episode IV - A New Hope (1977),Action|Adventure|Sci-Fi,4.099824
8,4993,576,"Lord of the Rings: The Fellowship of the Ring,...",Adventure|Fantasy,4.092134
9,47,568,Seven (a.k.a. Se7en) (1995),Mystery|Thriller,4.087199


In [79]:
###  Jaccard-Based Item Similarity
def get_jaccard_similar_items(seed_movie_id, top_n=10, min_common_users=5):
    if seed_movie_id not in movie_to_users:
        return pd.DataFrame()

    users_A = movie_to_users[seed_movie_id]
    jaccard_results = []

    for other_mid, users_B in movie_to_users.items():
        if other_mid == seed_movie_id:
            continue
        intersection = len(users_A & users_B)
        if intersection >= min_common_users:
            union = len(users_A | users_B)
            jaccard_sim = intersection / union
            jaccard_results.append({
                "movieId": other_mid,
                "common_users": intersection,
                "jaccard_similarity": round(jaccard_sim, 4)
            })

    res_df = pd.DataFrame(jaccard_results).sort_values("jaccard_similarity", ascending=False).head(top_n)
    return res_df.merge(movies[["movieId", "title", "genres", "average_rating"]], on="movieId")

print(f"Top 10 Jaccard-Similar Movies to '{seed_title}':")
display(get_jaccard_similar_items(seed_mid, top_n=10))

Top 10 Jaccard-Similar Movies to 'Shawshank Redemption, The (1994)':


,movieId,common_users,jaccard_similarity,title,genres,average_rating
0,296,870,0.4485,Pulp Fiction (1994),Comedy|Crime|Drama|Thriller,4.196969
1,356,851,0.4364,Forrest Gump (1994),Comedy|Drama|Romance|War,4.052744
2,593,805,0.4255,"Silence of the Lambs, The (1991)",Crime|Horror|Thriller,4.148367
3,50,679,0.3898,"Usual Suspects, The (1995)",Crime|Mystery|Thriller,4.265070
4,527,689,0.3851,Schindler's List (1993),Drama|War,4.236990
5,2571,759,0.3677,"Matrix, The (1999)",Action|Sci-Fi|Thriller,4.156437
6,2959,668,0.3625,Fight Club (1999),Action|Crime|Drama|Thriller,4.228780
7,47,568,0.3302,Seven (a.k.a. Se7en) (1995),Mystery|Thriller,4.087199
8,4993,576,0.3061,"Lord of the Rings: The Fellowship of the Ring,...",Adventure|Fantasy,4.092134
9,110,542,0.3030,Braveheart (1995),Action|Drama|War,3.988630


In [80]:
###  Optional Sparse KNN Experiment
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors

# Create Item-User Sparse Rating Matrix from training data
movie_cats = train_df["movieId"].astype("category")
user_cats = train_df["userId"].astype("category")

item_user_sparse = csr_matrix(
    (train_df["rating"].values, (movie_cats.cat.codes.values, user_cats.cat.codes.values)),
    shape=(len(movie_cats.cat.categories), len(user_cats.cat.categories))
)

mid_to_row = {mid: idx for idx, mid in enumerate(movie_cats.cat.categories)}
row_to_mid = dict(enumerate(movie_cats.cat.categories))

# Fit Cosine-based Sparse KNN
knn_item_model = NearestNeighbors(metric="cosine", algorithm="brute", n_neighbors=11)
knn_item_model.fit(item_user_sparse)

def get_knn_similar_movies(seed_movie_id, top_n=10):
    if seed_movie_id not in mid_to_row:
        return pd.DataFrame()
    row_idx = mid_to_row[seed_movie_id]
    distances, indices = knn_item_model.kneighbors(item_user_sparse[row_idx], n_neighbors=top_n + 1)

    knn_rows = []
    for dist, idx in zip(distances.flatten()[1:], indices.flatten()[1:]):
        knn_rows.append({
            "movieId": row_to_mid[idx],
            "cosine_distance": round(float(dist), 4),
            "cosine_similarity": round(1.0 - float(dist), 4)
        })
    return pd.DataFrame(knn_rows).merge(movies[["movieId", "title", "genres", "average_rating"]], on="movieId")

print(f"Sparse KNN Nearest Neighbors for '{seed_title}':")
display(get_knn_similar_movies(seed_mid, 10))

Sparse KNN Nearest Neighbors for 'Shawshank Redemption, The (1994)':


,movieId,cosine_distance,cosine_similarity,title,genres,average_rating
0,296,0.3390,0.6610,Pulp Fiction (1994),Comedy|Crime|Drama|Thriller,4.196969
1,356,0.3441,0.6559,Forrest Gump (1994),Comedy|Drama|Romance|War,4.052744
2,593,0.3610,0.6390,"Silence of the Lambs, The (1991)",Crime|Horror|Thriller,4.148367
3,527,0.3942,0.6058,Schindler's List (1993),Drama|War,4.236990
4,50,0.3961,0.6039,"Usual Suspects, The (1995)",Crime|Mystery|Thriller,4.265070
5,2571,0.4363,0.5637,"Matrix, The (1999)",Action|Sci-Fi|Thriller,4.156437
6,2959,0.4399,0.5601,Fight Club (1999),Action|Crime|Drama|Thriller,4.228780
7,47,0.4459,0.5541,Seven (a.k.a. Se7en) (1995),Mystery|Thriller,4.087199
8,110,0.4611,0.5389,Braveheart (1995),Action|Drama|War,3.988630
9,1704,0.4906,0.5094,Good Will Hunting (1997),Drama|Romance,4.091109


In [81]:
# ===  Statistical Evaluation (Precision@K and HitRate@K) ===

def recommend_via_item_similarity(user_id, method="jaccard", top_n=10):
    user_pos = positive_train[positive_train["userId"] == user_id].sort_values("rating", ascending=False)
    seen = set(train_df[train_df["userId"] == user_id]["movieId"])
    fallback = (
        movies[~movies["movieId"].isin(seen)]
        .sort_values("composite_score", ascending=False)["movieId"]
        .head(top_n)
        .tolist()
    )

    if user_pos.empty:
        return fallback

    anchor_movie = user_pos.iloc[0]["movieId"]

    if method == "cooccurrence":
        sim_df = get_cooccurrence_recommendations(anchor_movie, top_n=top_n + 20)

    elif method == "jaccard":
        if anchor_movie not in movie_to_users:
            return fallback
        users_A = movie_to_users[anchor_movie]
        jaccard_results = []
        for other_mid, users_B in movie_to_users.items():
            if other_mid == anchor_movie:
                continue
            intersection = len(users_A & users_B)
            if intersection >= 2:
                union = len(users_A | users_B)
                jaccard_results.append({
                    "movieId": other_mid,
                    "jaccard_similarity": intersection / union
                })
        if not jaccard_results:
            return fallback
        sim_df = (
            pd.DataFrame(jaccard_results)
            .sort_values("jaccard_similarity", ascending=False)
            .head(top_n + 20)
        )

    else:  # method == "knn"
        sim_df = get_knn_similar_movies(anchor_movie, top_n=top_n + 20)

    if sim_df is None or sim_df.empty or "movieId" not in sim_df.columns:
        return fallback

    recs = [m for m in sim_df["movieId"].tolist() if m not in seen]
    return recs[:top_n] if recs else fallback


def run_full_evaluation(eval_user_list, k=10):
    # Ground truth: movies actually liked (rating >= 3.5) in the held-out Test set
    ground_truth = (
        test_df[test_df["rating"] >= 3.5]
        .groupby("userId")["movieId"]
        .apply(set)
        .to_dict()
    )
    global_pop_ids = (
        movies.sort_values("composite_score", ascending=False)["movieId"]
        .head(k + 50)
        .tolist()
    )

    results_tracker = {
        "1. Popularity-Based Baseline":   {"precision": [], "hit": []},
        "2. User Genre-Affinity":         {"precision": [], "hit": []},
        "3. Item Co-occurrence":          {"precision": [], "hit": []},
        "4. Jaccard Item Similarity":     {"precision": [], "hit": []},
        "5. Sparse KNN Item Similarity":  {"precision": [], "hit": []},
    }

    for uid in eval_user_list:
        actual_liked = ground_truth.get(uid, set())
        if not actual_liked:
            continue

        seen_train = set(train_df[train_df["userId"] == uid]["movieId"])

        model_outputs = {
            "1. Popularity-Based Baseline":  [m for m in global_pop_ids if m not in seen_train][:k],
            "2. User Genre-Affinity":        recommend_genre_affinity(uid, top_n=k)["movieId"].tolist(),
            "3. Item Co-occurrence":         recommend_via_item_similarity(uid, method="cooccurrence", top_n=k),
            "4. Jaccard Item Similarity":    recommend_via_item_similarity(uid, method="jaccard", top_n=k),
            "5. Sparse KNN Item Similarity": recommend_via_item_similarity(uid, method="knn", top_n=k),
        }

        for model_name, recs in model_outputs.items():
            hits = len(set(recs) & actual_liked)
            results_tracker[model_name]["precision"].append(hits / k)
            results_tracker[model_name]["hit"].append(1.0 if hits > 0 else 0.0)

    eval_table = []
    for model_name, scores in results_tracker.items():
        eval_table.append({
            "Recommendation Model": model_name,
            "Evaluated Users": len(scores["hit"]),
            f"Precision@{k}": round(float(np.mean(scores["precision"])), 4),
            f"HitRate@{k}": round(float(np.mean(scores["hit"])), 4)
        })

    return pd.DataFrame(eval_table)


# Run evaluation on 200 users at K=10
final_eval_df = run_full_evaluation(eligible_users[:200], k=10)
print("=== Final Statistical Evaluation Comparison (K = 10) ===")
display(final_eval_df)

=== Final Statistical Evaluation Comparison (K = 10) ===


,Recommendation Model,Evaluated Users,Precision@10,HitRate@10
0,1. Popularity-Based Baseline,200,0.0880,0.490
1,2. User Genre-Affinity,200,0.0145,0.140
2,3. Item Co-occurrence,200,0.1060,0.540
3,4. Jaccard Item Similarity,200,0.0880,0.465
4,5. Sparse KNN Item Similarity,200,0.0885,0.475
